todo: create dummy schemas that only include patient and medications, then continue this notebook

## Synth-EHR demo
 This notebook demonstrates the process of generating patients, distributing their medical histories across a realistic medical database, retrieving information via AI agents, and evaluating agent performance. 

 In this demo, the output cells are already populated. I did this so that anyone can follow this process without needing to set up synthea or have a current OpenAI / Anthropic API key.

 The format will be code > output from code > explanation

## Step 0: Load required libraries for running agent workflows
Because we are using python to call our agents and run their workflows, we need to import certain software libraries. We do that below, feel free to completely ignore if you are just trying to understand the overall process.

In [ ]:

import json
from pathlib import Path
import os
import traceback
import sqlite3
from .rebuild_db import rebuild_db
from .schema_adapters import flat_v1, normalized_v1

## Step 1: Generate patients
As mentioned in the readme.md, the first step is to generate patients. Doing so requires downloading the synthea java application. 
So for this demonstration, we are going to use a pre-generated patient, Carter MacGyver. You can take a closer look at his record at mvp/demo_patients/json/b7e15ab8-7633-df03-b152-d68acebedb56/patient.json.
to get his entire medical history. For this tutorial, however, the thing to keep in mind is that Carter's entire medical record was generated from the ground up.
For each medical event that transpires, we also have the events that led to it. All contained in a single datastructure.

## Step 2: Create databases for housing patient information
Part of the utility of this project is that it can generate database layouts dynamically, and run AI workflows on these databases regardless of their exact layout.
We demonstrate this by running our workflow on 2 databases the contain the same data, but organized differently.
We create databases to house patient basic information, and medication history. In our flat database, each patient + medication history is stored as a single record in a table. In our normalized database, we have separate tables for patient information and medication history.


In [ ]:
PATIENT_JSON_DIRECTORY = r"mvp/demo_patients/json/b7e15ab8-7633-df03-b152-d68acebedb56/patient.json"
SCHEMA_NAMES = ["normalized_v1", "flat_v1"]
DATABASE_DIRECTORY = "mvp/demonstration_databases"
SCHEMA_PATH_PREFIX = "mvp/schemas"

db_paths = []
for schema in SCHEMA_NAMES:
    schema_path = Path(fr"{SCHEMA_PATH_PREFIX}/{schema}.sql")
    db_path = Path(fr"{DATABASE_DIRECTORY}/{schema}.db")
    if os.path.exists(db_path):
        os.remove(db_path)

    conn = sqlite3.connect(db_path)

    with open(schema_path, "r") as f:
        conn.executescript(f.read())

    conn.close()

    print(f"{schema} database rebuilt successfully.")

    db_paths.append(db_path)

## Step 3: Distribute patient data across databases
The previous step created the databases themselves, now we insert Carter MacGyver's information into each database.

In [ ]:
with open(PATIENT_JSON_DIRECTORY/"patient.json", "r") as file:
                carter = json.load(file)
for db in db_paths:
    conn = sqlite3.connect(db)
    conn.execute("PRAGMA foreign_keys = ON")
    cursor = conn.cursor()
    patient_id = carter["patient"]["id"]
    cursor.execute(
        """
        INSERT INTO patients
        VALUES (?, ?, ?, ?, ?, ?)
        """,
        (
            carter["patient"]["id"],
            carter["patient"]["firstName"],
            carter["patient"]["lastName"],
            carter["patient"]["birthdate"],
            carter["patient"]["deathdate"],
            carter["patient"]["gender"]
        )
    )